# **Data Ingestion - UK Land Registry Price Paid Data**

This notebook handles the ingestion of UK Land Registry property price data using PySpark.

**Objectives:**
- Configure SparkSession with optimization settings
- Load CSV data efficiently
- Define and validate schema
- Implement partitioning strategy
- Convert to Parquet format
- Perform initial data quality checks

In [1]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, count, when, isnan, year, month, mean, stddev, min as spark_min, max as spark_max
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## **1. SparkSession Configuration**

In [2]:
spark = SparkSession.builder \
    .appName("UK_Property_Prices_Analysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "100") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")

log4j = spark.sparkContext._jvm.org.apache.log4j
log4j.LogManager.getLogger("org.apache.spark.storage.MemoryStore").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.storage.BlockManager").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.scheduler.DAGScheduler").setLevel(log4j.Level.ERROR)

print(f"Spark Version: {spark.version}")
print(f"Spark Configurations:")
print(f"  - Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"  - Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"  - Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  - Adaptive Execution: {spark.conf.get('spark.sql.adaptive.enabled')}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 12:40:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.0
Spark Configurations:
  - Driver Memory: 8g
  - Executor Memory: 8g
  - Shuffle Partitions: 200
  - Adaptive Execution: true


## **2. Schema Definition**

In [3]:
schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("date_of_transfer", StringType(), True),
    StructField("postcode", StringType(), True),
    StructField("property_type", StringType(), True),
    StructField("old_new", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("paon", StringType(), True),
    StructField("saon", StringType(), True),
    StructField("street", StringType(), True),
    StructField("locality", StringType(), True),
    StructField("town_city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("county", StringType(), True),
    StructField("ppd_category", StringType(), True),
    StructField("record_status", StringType(), True)
])

print("Schema defined with 16 columns")

Schema defined with 16 columns


## **3. Data Ingestion**

In [4]:
data_path = "../land_registry_data/merged_land_registry.csv"

start_time = time.time()

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .option("mode", "DROPMALFORMED") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv(data_path)

load_time = time.time() - start_time

print(f"Data loaded in {load_time:.2f} seconds")
print(f"Total records: {df.count():,}")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

Data loaded in 0.87 seconds


Total records: 6,721,312
Number of partitions: 100


## **4. Initial Data Exploration**

In [5]:
df.printSchema()
df.show(10, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- date_of_transfer: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- old_new: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- paon: string (nullable = true)
 |-- saon: string (nullable = true)
 |-- street: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- town_city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- county: string (nullable = true)
 |-- ppd_category: string (nullable = true)
 |-- record_status: string (nullable = true)

+--------------------------------------+------+----------------+--------+-------------+-------+--------+--------------+-------+-----------------+--------------+---------+--------------------+--------------+------------+-------------+
|transaction_id                        |price |date_of_transfer|postcode|property_type|old_new|duration|pa

In [6]:
print("Dataset Statistics:")
print(f"Total Rows: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names:")
for i, col_name in enumerate(df.columns, 1):
    print(f"{i}. {col_name}")

Dataset Statistics:


Total Rows: 6,721,312
Total Columns: 16

Column Names:
1. transaction_id
2. price
3. date_of_transfer
4. postcode
5. property_type
6. old_new
7. duration
8. paon
9. saon
10. street
11. locality
12. town_city
13. district
14. county
15. ppd_category
16. record_status


## **5. Data Validation and Quality Checks**

In [7]:
from pyspark.sql.functions import trim, regexp_replace, to_date, to_timestamp

df_clean = df \
    .withColumn("price", col("price").cast("integer")) \
    .withColumn("date_of_transfer", to_date(to_timestamp(col("date_of_transfer"), "yyyy-MM-dd HH:mm"))) \
    .withColumn("postcode", trim(col("postcode"))) \
    .withColumn("property_type", trim(col("property_type"))) \
    .withColumn("old_new", trim(col("old_new"))) \
    .withColumn("duration", trim(col("duration")))

print("Data cleaning applied")
df_clean.printSchema()

Data cleaning applied
root
 |-- transaction_id: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- date_of_transfer: date (nullable = true)
 |-- postcode: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- old_new: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- paon: string (nullable = true)
 |-- saon: string (nullable = true)
 |-- street: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- town_city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- county: string (nullable = true)
 |-- ppd_category: string (nullable = true)
 |-- record_status: string (nullable = true)



In [8]:
null_counts = df_clean.select([count(when(col(c).isNull(), c)).alias(c) for c in df_clean.columns])
null_counts_pd = null_counts.toPandas().T
null_counts_pd.columns = ['Null_Count']
null_counts_pd['Percentage'] = (null_counts_pd['Null_Count'] / df_clean.count() * 100).round(2)
print("\nNull Value Analysis:")
print(null_counts_pd)


Null Value Analysis:
                  Null_Count  Percentage
transaction_id             0        0.00
price                      0        0.00
date_of_transfer           0        0.00
postcode               20718        0.31
property_type              0        0.00
old_new                    0        0.00
duration                   0        0.00
paon                       0        0.00
saon                 5841384       86.91
street                119414        1.78
locality             4145695       61.68
town_city                  0        0.00
district                   0        0.00
county                     0        0.00
ppd_category               0        0.00
record_status              0        0.00


In [9]:
df_clean.select("price").summary().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|           6721312|
|   mean| 387798.3259115185|
| stddev|1661765.3733740891|
|    min|                 1|
|    25%|            170000|
|    50%|            265000|
|    75%|            410000|
|    max|         900000000|
+-------+------------------+



In [10]:
price_stats = df_clean.select(
    spark_min("price").alias("min_price"),
    spark_max("price").alias("max_price"),
    mean("price").alias("mean_price"),
    stddev("price").alias("stddev_price")
).collect()[0]

print(f"\nPrice Statistics:")
print(f"Min Price: £{price_stats['min_price']:,}")
print(f"Max Price: £{price_stats['max_price']:,}")
print(f"Mean Price: £{price_stats['mean_price']:,.2f}")
print(f"Std Dev: £{price_stats['stddev_price']:,.2f}")


Price Statistics:
Min Price: £1
Max Price: £900,000,000
Mean Price: £387,798.33
Std Dev: £1,661,765.37


## **6. Data Distribution Analysis**

In [11]:
property_type_dist = df_clean.groupBy("property_type").count().orderBy("count", ascending=False)
print("\nProperty Type Distribution:")
property_type_dist.show()

old_new_dist = df_clean.groupBy("old_new").count().orderBy("count", ascending=False)
print("\nOld/New Distribution:")
old_new_dist.show()

duration_dist = df_clean.groupBy("duration").count().orderBy("count", ascending=False)
print("\nDuration Distribution:")
duration_dist.show()


Property Type Distribution:


+-------------+-------+
|property_type|  count|
+-------------+-------+
|            T|1816153|
|            S|1789997|
|            D|1562348|
|            F|1176800|
|            O| 376014|
+-------------+-------+


Old/New Distribution:


+-------+-------+
|old_new|  count|
+-------+-------+
|      N|6005050|
|      Y| 716262|
+-------+-------+


Duration Distribution:


+--------+-------+
|duration|  count|
+--------+-------+
|       F|5176020|
|       L|1545292|
+--------+-------+



In [12]:
df_clean = df_clean.withColumn("year", year("date_of_transfer"))
df_clean = df_clean.withColumn("month", month("date_of_transfer"))

yearly_dist = df_clean.groupBy("year").count().orderBy("year")
print("\nYearly Distribution:")
yearly_dist.show()


Yearly Distribution:


+----+-------+
|year|  count|
+----+-------+
|2019|1011880|
|2020| 896782|
|2021|1280440|
|2022|1074872|
|2023| 858018|
|2024| 918266|
|2025| 681054|
+----+-------+



## **7. Partitioning Strategy**

In [13]:
df_partitioned = df_clean.repartition(100, "year", "property_type")

print(f"Number of partitions after repartitioning: {df_partitioned.rdd.getNumPartitions()}")

Number of partitions after repartitioning: 100


## **8. Save to Parquet Format**

In [14]:
output_path = "../data/property_prices.parquet"

start_time = time.time()

df_partitioned.write \
    .mode("overwrite") \
    .partitionBy("year", "property_type") \
    .parquet(output_path)

save_time = time.time() - start_time

print(f"Data saved to Parquet in {save_time:.2f} seconds")
print(f"Output path: {output_path}")

Data saved to Parquet in 25.00 seconds
Output path: ../data/property_prices.parquet


## **9. Verify Parquet Data**

In [15]:
df_parquet = spark.read.parquet(output_path)

print(f"Parquet data loaded")
print(f"Total records: {df_parquet.count():,}")
print(f"Number of partitions: {df_parquet.rdd.getNumPartitions()}")
df_parquet.printSchema()

Parquet data loaded
Total records: 6,721,312
Number of partitions: 67
root
 |-- transaction_id: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- date_of_transfer: date (nullable = true)
 |-- postcode: string (nullable = true)
 |-- old_new: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- paon: string (nullable = true)
 |-- saon: string (nullable = true)
 |-- street: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- town_city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- county: string (nullable = true)
 |-- ppd_category: string (nullable = true)
 |-- record_status: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- property_type: string (nullable = true)



## **10. Performance Metrics**

In [16]:
print("\n=== Performance Summary ===")
print(f"CSV Load Time: {load_time:.2f} seconds")
print(f"Parquet Save Time: {save_time:.2f} seconds")
print(f"Total Records Processed: {df_partitioned.count():,}")
print(f"Final Partitions: {df_partitioned.rdd.getNumPartitions()}")


=== Performance Summary ===
CSV Load Time: 0.87 seconds
Parquet Save Time: 25.00 seconds


Total Records Processed: 6,721,312
Final Partitions: 100


In [17]:
sample_data = df_clean.limit(10000).toPandas()
sample_data.to_csv("../data/samples/sample_10k.csv", index=False)
print("Sample data saved")

Sample data saved
